In [1]:
import pandas as pd
import numpy as np
import re

from pathlib import Path
from itertools import combinations
from collections import defaultdict

# Project paths
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load data
df = pd.read_parquet(PROCESSED_DIR / "recipes.parquet")
ingr = pd.read_parquet(PROCESSED_DIR / "ingredient_embeddings.parquet")

print("Number of recipes:", len(df))
print("Number of unique embedded ingredients:", len(ingr))

Number of recipes: 228542
Number of unique embedded ingredients: 13598


In [2]:
ingredients = ingr["ingredient"].tolist()

ingredient_vectors = np.array(
    ingr["embedding"].tolist(),
    dtype=np.float32
)

print("Unique ingredients:", len(ingredients))
print("First 20 ingredients:", ingredients[:20])


Unique ingredients: 13598
First 20 ingredients: ['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'butter', 'olive oil', 'salt', 'prepared pizza crust', 'sausage patty', 'egg', 'milk', 'pepper', 'cheese', 'ground beef', 'yellow onion', 'diced tomato', 'tomato paste', 'tomato soup', 'rotel tomato', 'kidney bean']


In [3]:
def normalize_ingredient(ingredient):
    ingredient = ingredient.lower().strip()

    # Normalize whitespace
    ingredient = re.sub(r"\s+", " ", ingredient)

    # Normalize hyphens and apostrophes
    ingredient = ingredient.replace("-", " ")
    ingredient = ingredient.replace("'", "")

    # Remove unnecessary punctuation
    ingredient = re.sub(r"[^\w\s]", "", ingredient)

    return ingredient.strip()


In [4]:
threshold = 0.75
chunk_size = 256

matches = []
n = len(ingredients)

for start in range(0, n, chunk_size):

    end = min(start + chunk_size, n)

    similarities = (
        ingredient_vectors[start:end] @ ingredient_vectors.T
    )

    for local_i in range(end - start):

        i = start + local_i

        # Only retain pairs where i < j
        similarities[local_i, :i + 1] = -1

        indices = np.where(
            similarities[local_i] > threshold
        )[0]

        for j in indices:

            matches.append({
                "ingredient_1": ingredients[i],
                "ingredient_2": ingredients[j],
                "similarity": float(similarities[local_i, j])
            })

matches_df = pd.DataFrame(matches)

print("Candidate pairs:", len(matches_df))


Candidate pairs: 45928


In [5]:

def token_jaccard(a, b):
    tokens_a = set(normalize_ingredient(a).split())
    tokens_b = set(normalize_ingredient(b).split())

    union = tokens_a | tokens_b

    if not union:
        return 1.0

    return len(tokens_a & tokens_b) / len(union)


def character_shingles(text, n=3):
    text = normalize_ingredient(text)

    if len(text) < n:
        return {text}

    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }


def character_jaccard(a, b, n=3):
    shingles_a = character_shingles(a, n)
    shingles_b = character_shingles(b, n)

    union = shingles_a | shingles_b

    if not union:
        return 1.0

    return len(shingles_a & shingles_b) / len(union)


matches_df["token_jaccard"] = matches_df.apply(
    lambda row: token_jaccard(
        row["ingredient_1"],
        row["ingredient_2"]
    ),
    axis=1
)

matches_df["char_trigram"] = matches_df.apply(
    lambda row: character_jaccard(
        row["ingredient_1"],
        row["ingredient_2"]
    ),
    axis=1
)


In [6]:
matches_df["token_jaccard"] = matches_df.apply(
    lambda row: token_jaccard(
        row["ingredient_1"],
        row["ingredient_2"]
    ),
    axis=1
)

matches_df["char_bigram"] = matches_df.apply(
    lambda row: character_jaccard(
        row["ingredient_1"],
        row["ingredient_2"],
        n=2
    ),
    axis=1
)

matches_df["char_trigram"] = matches_df.apply(
    lambda row: character_jaccard(
        row["ingredient_1"],
        row["ingredient_2"],
        n=3
    ),
    axis=1
)

In [7]:
similarity_columns = [
    "similarity",
    "token_jaccard",
    "char_bigram",
    "char_trigram"
]

correlations = matches_df[similarity_columns].corr()

display(correlations)

,similarity,token_jaccard,char_bigram,char_trigram
similarity,1.000000,0.578010,0.585678,0.573647
token_jaccard,0.578010,1.000000,0.728650,0.727351
char_bigram,0.585678,0.728650,1.000000,0.957057
char_trigram,0.573647,0.727351,0.957057,1.000000


Then I created to files where I manually decided if one would merge a pair or not. These are the ingredient_validation.csv and ingredient_test_set.csv file.
Unfortunaltely I couldn't decide on a suitable combined threshold approach as can be seen below

Due to the high correltion of two and three character shingles (which can also be seen in the plots) I only used char_trigram

In [8]:
validation = pd.read_csv("ingredient_validation.csv")

validation["merge_label"] = pd.to_numeric(
    validation["merge_label"],
    errors="coerce"
)

validation = validation.dropna(
    subset=["merge_label"]
)

validation["merge_label"] = validation["merge_label"].astype(int)

In [10]:
results = []

for cosine_threshold in [0.80, 0.85, 0.90, 0.93, 0.95, 0.97]:
    for token_threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
        for trigram_threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:

            predicted = (
                (validation["similarity"] >= cosine_threshold) &
                (validation["token_jaccard"] >= token_threshold) &
                (validation["char_trigram"] >= trigram_threshold)
            )

            actual = validation["merge_label"].astype(bool)

            true_positive = (predicted & actual).sum()
            false_positive = (predicted & ~actual).sum()
            false_negative = (~predicted & actual).sum()

            precision = (
                true_positive / (true_positive + false_positive)
                if true_positive + false_positive > 0 else 0
            )

            recall = (
                true_positive / (true_positive + false_negative)
                if true_positive + false_negative > 0 else 0
            )

            results.append({
                "cosine_threshold": cosine_threshold,
                "token_threshold": token_threshold,
                "trigram_threshold": trigram_threshold,
                "precision": precision,
                "recall": recall,
                "merges": int(predicted.sum()),
                "false positive": false_positive
            })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    ["merges", "precision", "recall"],
    ascending=[False, False, False]
)

print(results_df.head(10).to_string(index=False))


 cosine_threshold  token_threshold  trigram_threshold  precision   recall  merges  false positive
             0.80              0.3                0.3   0.611111 0.814815     144              56
             0.80              0.4                0.3   0.628788 0.768519     132              49
             0.80              0.3                0.4   0.613636 0.750000     132              51
             0.80              0.4                0.4   0.617886 0.703704     123              47
             0.80              0.5                0.3   0.614754 0.694444     122              47
             0.80              0.5                0.4   0.612069 0.657407     116              45
             0.85              0.3                0.3   0.672727 0.685185     110              36
             0.80              0.3                0.5   0.657407 0.657407     108              37
             0.85              0.3                0.4   0.676190 0.657407     105              34
             0.85   

In [11]:
validation_test = pd.read_csv("ingredient_test_set.csv")

validation_test["merge_label"] = pd.to_numeric(
    validation_test["merge_label"],
    errors="coerce"
)

validation_test = validation_test.dropna(
    subset=["merge_label"]
)

validation_test["merge_label"] = validation_test["merge_label"].astype(int)

In [23]:
test_results = []

for cosine_threshold in [0.80, 0.85, 0.90, 0.93, 0.95, 0.97]:
    for token_threshold in [0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
        for trigram_threshold in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
            
            
            predicted = (
                ((validation_test["similarity"] >= cosine_threshold) &
                (validation_test["token_jaccard"] >= token_threshold)) |
                ((validation_test["similarity"] >= cosine_threshold) &
                 (validation_test["char_trigram"] >= trigram_threshold)) #|
                #((validation_test["token_jaccard"] >= token_threshold) &
                # (validation_test["char_trigram"] >= trigram_threshold))
                
            )

            actual = validation_test["merge_label"].astype(bool)

            true_positive = (predicted & actual).sum()
            false_positive = (predicted & ~actual).sum()
            false_negative = (~predicted & actual).sum()

            precision = (
                true_positive / (true_positive + false_positive)
                if true_positive + false_positive > 0 else 0
            )

            recall = (
                true_positive / (true_positive + false_negative)
                if true_positive + false_negative > 0 else 0
            )

            test_results.append({
                "cosine_threshold": cosine_threshold,
                "token_threshold": token_threshold,
                "trigram_threshold": trigram_threshold,
                "precision": precision,
                "recall": recall,
                "merges": int(predicted.sum()),
                "false positive": false_positive
            })

test_results_df = pd.DataFrame(test_results)

test_results_df = test_results_df.sort_values(
    ["merges", "precision", "recall"],
    ascending=[False, False, False]
)

print(test_results_df.head(15).to_string(index=False))


 cosine_threshold  token_threshold  trigram_threshold  precision   recall  merges  false positive
              0.8              0.4                0.3   0.441860 0.617886     172              96
              0.8              0.5                0.3   0.435294 0.601626     170              96
              0.8              0.6                0.3   0.439759 0.593496     166              93
              0.8              0.7                0.3   0.439759 0.593496     166              93
              0.8              0.8                0.3   0.439759 0.593496     166              93
              0.8              0.9                0.3   0.439759 0.593496     166              93
              0.8              0.4                0.4   0.464968 0.593496     157              84
              0.8              0.5                0.4   0.466216 0.560976     148              79
              0.8              0.4                0.5   0.476190 0.569106     147              77
              0.8   

In [13]:
test_results_similarity = []

for threshold in [0.80, 0.85, 0.90, 0.93, 0.95, 0.97, 0.98]:

    predicted = validation_test["similarity"] >= threshold
    actual = validation_test["merge_label"].astype(bool)

    true_positive = (predicted & actual).sum()
    false_positive = (predicted & ~actual).sum()
    false_negative = (~predicted & actual).sum()

    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive > 0 else 0
    )

    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative > 0 else 0
    )

    test_results_similarity.append({
        "cosine_threshold": threshold,
        "token_threshold": "",
        "trigram_threshold": "",
        "precision": precision,
        "recall": recall,
        "merges": int(predicted.sum()),
        "false positive": false_positive
    })

test_results_similarity_df = pd.DataFrame(test_results_similarity)

test_results_similarity_df = test_results_similarity_df.sort_values(
    ["merges", "precision", "recall"],
    ascending=[False, False, False]
)

print(test_results_similarity_df.head(20).to_string(index=False))

 cosine_threshold token_threshold trigram_threshold  precision   recall  merges  false positive
             0.80                                     0.436842 0.674797     190             107
             0.85                                     0.518519 0.341463      81              39
             0.90                                     0.538462 0.113821      26              12
             0.93                                     0.625000 0.040650       8               3
             0.95                                     0.750000 0.024390       4               1
             0.97                                     0.000000 0.000000       0               0
             0.98                                     0.000000 0.000000       0               0


In [14]:
test_results_tokens = []

for threshold in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:

    predicted = validation_test["token_jaccard"] >= threshold
    actual = validation_test["merge_label"].astype(bool)

    true_positive = (predicted & actual).sum()
    false_positive = (predicted & ~actual).sum()
    false_negative = (~predicted & actual).sum()

    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive > 0 else 0
    )

    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative > 0 else 0
    )

    test_results_tokens.append({
        "cosine_threshold": "",
        "token_threshold": threshold,
        "trigram_threshold": "",
        "precision": precision,
        "recall": recall,
        "merges": int(predicted.sum()),
        "false positive": false_positive
    })

test_results_tokens_df = pd.DataFrame(test_results_tokens)

test_results_tokens_df = test_results_tokens_df.sort_values(
    ["merges", "precision", "recall"],
    ascending=[False, False, False]
)

print(test_results_tokens_df.head(20).to_string(index=False))

cosine_threshold  token_threshold trigram_threshold  precision   recall  merges  false positive
                              0.3                     0.347541 0.861789     305             199
                              0.4                     0.385281 0.723577     231             142
                              0.5                     0.398907 0.593496     183             110
                              0.6                     0.555556 0.284553      63              28
                              0.7                     0.428571 0.024390       7               4
                              0.8                     0.000000 0.000000       0               0
                              0.9                     0.000000 0.000000       0               0


In [16]:
test_results_3char = []

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:

    predicted = validation_test["char_trigram"] >= threshold
    actual = validation_test["merge_label"].astype(bool)

    true_positive = (predicted & actual).sum()
    false_positive = (predicted & ~actual).sum()
    false_negative = (~predicted & actual).sum()

    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive > 0 else 0
    )

    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative > 0 else 0
    )

    test_results_3char.append({
        "cosine_threshold": "",
        "token_threshold": "",
        "trigram_threshold": threshold,
        "precision": precision,
        "recall": recall,
        "merges": int(predicted.sum()),
        "false positive": false_positive
    })

test_results_3char_df = pd.DataFrame(test_results_3char)

test_results_3char_df = test_results_3char_df.sort_values(
    ["merges", "precision", "recall"],
    ascending=[False, False, False]
)

print(test_results_3char_df.head(20).to_string(index=False))

cosine_threshold token_threshold  trigram_threshold  precision   recall  merges  false positive
                                                0.2   0.313673 0.951220     373             256
                                                0.3   0.341297 0.813008     293             193
                                                0.4   0.416216 0.626016     185             108
                                                0.5   0.526786 0.479675     112              53
                                                0.6   0.604651 0.211382      43              17
                                                0.7   0.684211 0.105691      19               6
                                                0.8   0.750000 0.024390       4               1
                                                0.9   0.000000 0.000000       0               0


In [28]:
good_rules = test_results_df[
    (test_results_df["precision"] >= 0.55) |
    (test_results_df["merges"] >= 50) |
    (test_results_df["false positive"] < 5)
].sort_values(
    ["false positive", "recall", "merges"],
    ascending=[True, False, False]
)

print(good_rules.head(10).to_string(index=False))

 cosine_threshold  token_threshold  trigram_threshold  precision  recall  merges  false positive
             0.90              0.8                0.8        1.0 0.01626       2               0
             0.90              0.9                0.8        1.0 0.01626       2               0
             0.95              0.6                0.8        1.0 0.00813       1               0
             0.95              0.6                0.9        1.0 0.00813       1               0
             0.80              0.8                0.9        0.0 0.00000       0               0
             0.80              0.9                0.9        0.0 0.00000       0               0
             0.85              0.8                0.9        0.0 0.00000       0               0
             0.85              0.9                0.9        0.0 0.00000       0               0
             0.90              0.8                0.9        0.0 0.00000       0               0
             0.90             

In [19]:
strong_candidates = matches_df[
    (matches_df["similarity"] >= 0.97) &
    (matches_df["token_jaccard"] >= 0.90) &
    (matches_df["char_trigram"] >= 0.90)
].copy()

broader_candidates = matches_df[
    (matches_df["similarity"] >= 0.97) |
    (matches_df["token_jaccard"] >= 0.90) |
    (matches_df["char_trigram"] >= 0.90)
].copy()

print("All candidate pairs:", len(matches_df))
print("Strong candidates (AND):", len(strong_candidates))
print("Broader candidates (OR):", len(broader_candidates))

All candidate pairs: 45928
Strong candidates (AND): 24
Broader candidates (OR): 213


Since only very few candidates fulfill this conservative condition the effect is considered to be negletible and hence no real merging is done and we stick to the one of the preprocessing.